# Notebook: BCI_21_CarDet_Evalua_Datos_Exc
*********************************************************************************

## Informacion del Notebook

### Encabezado
**************************************************************************
* Nombre: BCI_21_CarDet_Evalua_Datos_Exc.ipynb
* Ruta: https://adb-5512273708018582.2.azuredatabricks.net/?o=5512273708018582#notebook/2997520011899624
* Autor: Gabriel Martínez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 12/08/2022
* Descripcion: Evaluacion criterior de excepciones a entrada a deterioro
* Documentacion:
***************************************************************************

### Mantenciones
**************************************************************************
#### Mantención Nro: 1
* Autor: Gagriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 10/02/2025 
* Descripción: Se modifica el criterio 27 no considere las filiales.    
***************************************************************************

### Tablas Entrada y Salida
**************************************************************************
#### Tablas Entrada: 
* {base_silver_x}.tbl_cd_ope_condicion_deterioro
***************************************************************************
#### Tablas Salida: 
* {base_silver_x}.tbl_cd_cartdet_crit_ent_ope_eval
***************************************************************************


## Carga Dependencias

### Setea Parámetros

In [0]:

dbutils.widgets.text("fecha_w","","01-Fecha:")
dbutils.widgets.text("bd_silver_w","","02-Nombre BD Silver:")

fecha_x = dbutils.widgets.get("fecha_w") 
base_silver_x = dbutils.widgets.get("bd_silver_w")

spark.conf.set("bci.fecha", fecha_x)
spark.conf.set("bci.dbnamesilver", base_silver_x)

print(f"Fecha de Proceso actual: [fecha_x] {fecha_x}")
print(f"Nombre BD Silver: [base_silver_x] {base_silver_x}")



Fecha de Proceso actual: [fecha_x] 20250930
Nombre BD Silver: [base_silver_x] dsr_gld_bciwork_db


### Carga funciones comunes

In [0]:
%run "./Funciones_Comunes"

# Notebook: Funciones_Comunes
**************************************************************************

## Informacion del Notebook 

### Encabezado
**************************************************************************
* Nombre: Funciones_Comunes.ipynb
* Ruta: https://adb-5512273708018582.2.azuredatabricks.net/?o=5512273708018582#notebook/3570959530595695
* Autor: Gabriel Martínez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 23/09/2023
* Descripcion: Notebook con funciones genéricas que pueden ser usadas por otros notebooks.
* Documentacion:
***************************************************************************

### Mantenciones
**************************************************************************
#### Mantención Nro: 1
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 15/02/2025 
* Descripción: Se cambio el metodo de cancelacion utilizando el comando (raise) y se incorporada la funcion de ir a buscar el ultimo dia calendario. Tambien se agrego una nueva funcion (obtener_estados_tablas).  
***************************************************************************
#### Mantención Nro: 2
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 25/04/2025 
* Descripción: Se modifico la funcion extension_archivos para que cuando la vigencia sea previa, asigne extencion .PRV.  
***************************************************************************
#### Mantención Nro: 3
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Claudia Yañez
* Fecha: 08/07/2025 
* Descripción: Se realiza una mejora en la funcion mostrar_variacion_criterio.  
***************************************************************************

## Carga librerias

...

### obtener archivo periodo anterior

### concatena archivos

###primer_dia_mes_sig

###Calcula fecha X meses atras

## FIN definición de funciones

### Valida parámetros

In [0]:
valida_parametro(fecha_x)

In [0]:
valida_parametro(base_silver_x)

## INICIO Proceso extraccion y transformacion
--------------------------------------
- Por cada fuente que se utilice se debe:
     - Titulo: generar un titulo generico, con nombre fuente, y descripcion del proposito de la extraccion
     - Extraer: para el periodo, o rango de fecha que se necesita la iformacion. Debe tener el prefijo tmp_EXT_{nombrefuente}
     - Transformar: generar la informacion necesaria para la salida final. Se pueden generar mas de una tabla temporal para llegar al resultado final. Debe tener el prefijo tmp_RES_{nombre}_correlativo


### Parametrizacion
---
* define y asigna valores a los parametros


In [0]:
#parametria interna notebook
p_cod_evaluacion='N01','N02'
print(f"p_cod_evaluacion: {p_cod_evaluacion}")

p_cod_evaluacion: ('N01', 'N02')


In [0]:
#Para el cálculo de clientes LIR se debe tomar los ultimos 12 meses a partir de la fecha de proceso
p_fec_12_meses_atras=fecha_x_meses_atras(fecha_x,12)
print(f"p_fec_12_meses_atras: {p_fec_12_meses_atras}")

p_fec_12_meses_atras: 20240930


### Evaluacion: operaciones con monto deuda ifrs cero
---
* operaciones con monto dueda total ifrs cero
* Excepto para operaciones Nova CCN455, COLMN CON731 Y COLMN COM732


In [0]:
paso_query20 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_1 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'saldo_total_ifrs-sistema-tipo_operacion-flag_existe_cliente_lir-cli_fecha_informada_lir-flag_existe_cliente_ssff'               AS nombre_campo,
    concat('[',cast(trim(A.saldo_total_ifrs) as string),'-',cast(trim(A.sistema) as string),'-',cast(trim(A.tipo_operacion) as string),'-', cast(A.flag_existe_cliente_lir as string),'-', cast(A.cli_fecha_informada_lir as string),'-', cast(A.flag_existe_cliente_ssff as string),']')        AS valor_campo,
    "Excepcion saldo ifrs = 0, para operaciones del sitema y tioaux a 20-CCN455, 01-CON731, 01-COM732, y cliente no lir y cliente no ssff"                     AS condicion_regla,     
    concat('[','0','-','(20-CCN455, 01-CON731, 01-COM732)','-','NO_LIR','-','NO_SSFF',']')               AS valor_regla,           
    CASE 
       WHEN trim(A.sistema) = '20' AND trim(A.tipo_operacion)='CCN455' THEN 0
       WHEN trim(A.sistema) = '01' AND trim(A.tipo_operacion)='CON731' THEN 0
       WHEN trim(A.sistema) = '01' AND trim(A.tipo_operacion)='COM732' THEN 0
       WHEN A.flag_existe_cliente_lir = 1 and A.cli_fecha_informada_lir > {p_fec_12_meses_atras}  THEN 0
       WHEN A.flag_existe_cliente_ssff =1  THEN 0
       ELSE 1 
    END                                                          AS flag_resultado_regla,
    'N01'                                                        AS cod_evaluacion  
FROM 
    {base_silver_x}.tbl_cd_ope_condicion_deterioro   A
WHERE 
    IFNULL(A.saldo_total_ifrs,0) = 0    
"""

In [0]:
sql_safe(paso_query20)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_1 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'saldo_total_ifrs-sistema-tipo_operacion-flag_existe_cliente_lir-cli_fecha_informada_lir-flag_existe_cliente_ssff'               AS nombre_campo,
    concat('[',cast(trim(A.saldo_to

DataFrame[]

### Evaluacion: operaciones cae excepcion
---
* operaciones CAE excepcion informadas en archivo


In [0]:
paso_query40 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_2 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'operacion-tipo_operacion'                                   AS nombre_campo,
    concat('[',cast(A.operacion as string),'-',cast(A.tipo_operacion as string),']')  AS valor_campo,
    "Excepcion CAE operacion y tipo de operacion mismo agno de licitacion"        AS condicion_regla,      
    Case when A.flag_cae_excepcion = 1 THEN 'EXISTE (detalle_incumplimiento_cierre_cae)' ELSE 'NO_EXISTE (detalle_incumplimiento_cierre_cae)'  END AS valor_regla,
    CASE 
         WHEN A.flag_cae_excepcion = 1
         THEN 1 
         ELSE 0 
     END                                                         AS flag_resultado_regla,
    'N02'                                                        AS cod_evaluacion  
FROM 
    {base_silver_x}.tbl_cd_ope_condicion_deterioro   A
WHERE
     A.flag_cae_excepcion = 1    
"""

In [0]:
sql_safe(paso_query40)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CAMPO_EVAL_2 as
SELECT
    A.periodo_cierre                                            AS periodo_cierre,
    A.fecha_cierre                                              AS fecha_cierre,
    A.tipo_proceso                                              AS tipo_proceso,
    A.segmento                                                  AS segmento,
    A.operacion                                                 AS operacion,
    A.tipo_operacion                                            AS tipo_operacion,
    A.sistema                                                   AS sistema,
    A.rut_cliente                                               AS rut_cliente,
    A.dv_rut_cliente                                            AS dv_rut_cliente ,
    'operacion-tipo_operacion'                                   AS nombre_campo,
    concat('[',cast(A.operacion as string),'-',cast(A.tipo_operacion as string),']')  AS valor_campo,
 

DataFrame[]

### Salida Temporal a Nivel de Campo Evaludado (tmp_tbl_cartdet_crit_ent_ope_campo)
------------------
* generar salida temporal a nivel de campo evaluado. 
* se registran todas las operaciones evaluadas


In [0]:

paso_query250 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_tbl_cartdet_crit_ent_ope_eval AS
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_1 
UNION 
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_2 
"""  

In [0]:
sql_safe(paso_query250)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_tbl_cartdet_crit_ent_ope_eval AS
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_1 
UNION 
SELECT * FROM   tmp_RES_D00_OPE_CAMPO_EVAL_2 



DataFrame[]

## Carga Tablas de Salidas
--------------------------------------
* carga resultados a tablas de salidas del notebook

### Carga Tabla Evaluacion 


#### Reproceso (Elimina registros en caso de reprocesos). Tabla no es historica

In [0]:
paso_query300 = f"""DELETE FROM {base_silver_x}.tbl_cd_cartdet_crit_ent_ope_eval where cod_evaluacion in {p_cod_evaluacion} """

In [0]:
sql_safe(paso_query300)

sql_safe: query -> DELETE FROM dsr_gld_bciwork_db.tbl_cd_cartdet_crit_ent_ope_eval where cod_evaluacion in ('N01', 'N02') 


DataFrame[num_affected_rows: bigint]

#### Inserta Registros tabla salida

In [0]:

paso_query310 = f"""
INSERT INTO {base_silver_x}.tbl_cd_cartdet_crit_ent_ope_eval
SELECT 
    IFNULL(periodo_cierre,19000101),
    IFNULL(fecha_cierre,190001),
    IFNULL(tipo_proceso,' '),
    IFNULL(segmento,' '),
    IFNULL(operacion,' '),
    IFNULL(tipo_operacion,' '),
    IFNULL(sistema,' '),
    IFNULL(rut_cliente,0),
    IFNULL(dv_rut_cliente,' '),
    IFNULL(nombre_campo,' '),
    IFNULL(valor_campo,' '),
    IFNULL(condicion_regla,' '),
    IFNULL(valor_regla,' '),
    IFNULL(flag_resultado_regla,0),
    IFNULL(cod_evaluacion,' ')
FROM
  tmp_tbl_cartdet_crit_ent_ope_eval
"""  


In [0]:
sql_safe(paso_query310)

sql_safe: query -> 
INSERT INTO dsr_gld_bciwork_db.tbl_cd_cartdet_crit_ent_ope_eval
SELECT 
    IFNULL(periodo_cierre,19000101),
    IFNULL(fecha_cierre,190001),
    IFNULL(tipo_proceso,' '),
    IFNULL(segmento,' '),
    IFNULL(operacion,' '),
    IFNULL(tipo_operacion,' '),
    IFNULL(sistema,' '),
    IFNULL(rut_cliente,0),
    IFNULL(dv_rut_cliente,' '),
    IFNULL(nombre_campo,' '),
    IFNULL(valor_campo,' '),
    IFNULL(condicion_regla,' '),
    IFNULL(valor_regla,' '),
    IFNULL(flag_resultado_regla,0),
    IFNULL(cod_evaluacion,' ')
FROM
  tmp_tbl_cartdet_crit_ent_ope_eval



DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

##Estadisticas tabla salida

In [0]:
%sql
SELECT
fecha_cierre,
cod_evaluacion,
CASE 
  WHEN cod_evaluacion='E01' THEN 'CLIENTE_INDIVIDUAL_DETERIORADO'
  WHEN cod_evaluacion='E04' THEN 'CLIENTE_CON_MOROSIDAD'  
  WHEN cod_evaluacion='E05' THEN 'CLIENTE_CON_RENEGOCIADO'    
  WHEN cod_evaluacion='E06' THEN 'CLIENTE_REESTRUCTURCION_FORZOSA'    
  WHEN cod_evaluacion='E07' THEN 'CLIENTE_LIR'    
  WHEN cod_evaluacion='E08' THEN 'CLIENTE_DETERIORADO_SSFF'    
  WHEN cod_evaluacion='E09' THEN 'CLIENTE_DETERIORADO_FACT'
  WHEN cod_evaluacion='N01' THEN 'EXCEPCION_OPERACION_SALDO_IFRS_CERO'
  WHEN cod_evaluacion='N02' THEN 'EXCEPCION_OPERACION_CAE'
  ELSE 'NO_IDENTIFICADO'    
END                    AS des_cod_evaluacion,
COUNT(1) AS CANT_REG
FROM ${bci.dbnamesilver}.tbl_cd_cartdet_crit_ent_ope_eval
GROUP BY 1,2,3
ORDER BY 1,2,3


fecha_cierre,cod_evaluacion,des_cod_evaluacion,CANT_REG
20250930,E01,CLIENTE_INDIVIDUAL_DETERIORADO,3693104
20250930,E04,CLIENTE_CON_MOROSIDAD,3693104
20250930,E05,CLIENTE_CON_RENEGOCIADO,3693104
20250930,E06,CLIENTE_REESTRUCTURCION_FORZOSA,3693104
20250930,E07,CLIENTE_LIR,3693104
20250930,E08,CLIENTE_DETERIORADO_SSFF,3693104
20250930,E09,CLIENTE_DETERIORADO_FACT,3693104
20250930,N01,EXCEPCION_OPERACION_SALDO_IFRS_CERO,27554
20250930,N02,EXCEPCION_OPERACION_CAE,22


In [0]:
%sql
SELECT
fecha_cierre,
cod_evaluacion,
flag_resultado_regla,
CASE 
  WHEN cod_evaluacion='E01' THEN 'CLIENTE_INDIVIDUAL_DETERIORADO'
  WHEN cod_evaluacion='E04' THEN 'CLIENTE_CON_MOROSIDAD'  
  WHEN cod_evaluacion='E05' THEN 'CLIENTE_CON_RENEGOCIADO'    
  WHEN cod_evaluacion='E06' THEN 'CLIENTE_REESTRUCTURCION_FORZOSA'    
  WHEN cod_evaluacion='E07' THEN 'CLIENTE_LIR'    
  WHEN cod_evaluacion='E08' THEN 'CLIENTE_DETERIORADO_SSFF'    
  WHEN cod_evaluacion='E09' THEN 'CLIENTE_DETERIORADO_FACT'
  WHEN cod_evaluacion='N01' THEN 'EXCEPCION_OPERACION_SALDO_IFRS_CERO'
  WHEN cod_evaluacion='N02' THEN 'EXCEPCION_OPERACION_CAE'
  ELSE 'NO_IDENTIFICADO'    
END                    AS des_cod_evaluacion,
COUNT(1) AS CANT_REG
FROM ${bci.dbnamesilver}.tbl_cd_cartdet_crit_ent_ope_eval
GROUP BY 1,2,3,4
ORDER BY 1,2,3,4


fecha_cierre,cod_evaluacion,flag_resultado_regla,des_cod_evaluacion,CANT_REG
20250930,E01,0,CLIENTE_INDIVIDUAL_DETERIORADO,3690642
20250930,E01,1,CLIENTE_INDIVIDUAL_DETERIORADO,2462
20250930,E04,0,CLIENTE_CON_MOROSIDAD,3497353
20250930,E04,1,CLIENTE_CON_MOROSIDAD,195751
20250930,E05,0,CLIENTE_CON_RENEGOCIADO,3692367
20250930,E05,1,CLIENTE_CON_RENEGOCIADO,737
20250930,E06,0,CLIENTE_REESTRUCTURCION_FORZOSA,3693017
20250930,E06,1,CLIENTE_REESTRUCTURCION_FORZOSA,87
20250930,E07,0,CLIENTE_LIR,3683588
20250930,E07,1,CLIENTE_LIR,9516


## Mensaje termino OK

In [0]:
msgerrorx="OK"
dbutils.notebook.exit("{\"coderror\":0, \"msgerror\":\""+msgerrorx+"\"}")

In [0]:
%sql
select * from dsr_gld_bciwork_db.tbl_cd_cartdet_crit_ent_ope_eval where rut_cliente=17944363